<a href="https://colab.research.google.com/github/Lilian-Santtos/atividade-avaliativa-processamento-dados-massivos-pyspark/blob/main/Lista_Exercicios_1_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Processamento de Dados Massivos

## Lista de Exercícios 1 — Introdução ao PySpark

**Aluna:** Lilian Santos  
**Professor:** Alexandre Roriz  
**Curso:** Ciência de Dados e Inteligência Artificial — IESB

## Preparação do Ambiente

Nesta etapa, será realizada a preparação do ambiente PySpark, o download da base de dados e a criação do DataFrame que será utilizado nas questões seguintes.

In [16]:
!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/nyc_tripdata_2024_sample_4M.csv

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("ExerciciosPySpark")
    .master("local[*]")
    .getOrCreate()
)

df = spark.read.csv(
    "nyc_tripdata_2024_sample_4M.csv",
    header=True,
    inferSchema=True
)

## Questão 1

Nesta questão, serão exibidos o schema inferido do DataFrame, as 10 primeiras linhas e o número total de registros do dataset.

In [17]:
# a) Exibir o schema inferido do DataFrame
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: integer (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: integer (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: integer (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [18]:
# b) Exibir as 10 primeiras linhas do DataFrame
df.show(10)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2024-10-01 00:59:55|  2024-10-01 01:02:24|              1|          0.5|         1|                 N|         230|         161|           1|        5.1|  3.5|    0.5|       2.

In [19]:
# c) Exibir o número total de linhas do dataset
df.count()

4118743

## Questão 2

Nesta questão, serão selecionadas apenas as colunas VendorID, tpep_pickup_datetime, trip_distance, fare_amount e payment_type, exibindo as 5 primeiras linhas do resultado.

In [20]:
# Selecionar as colunas solicitadas e exibir as 5 primeiras linhas

df.select(
    "VendorID",
    "tpep_pickup_datetime",
    "trip_distance",
    "fare_amount",
    "payment_type"
).show(5)

+--------+--------------------+-------------+-----------+------------+
|VendorID|tpep_pickup_datetime|trip_distance|fare_amount|payment_type|
+--------+--------------------+-------------+-----------+------------+
|       1| 2024-10-01 00:59:55|          0.5|        5.1|           1|
|       1| 2024-10-01 00:08:59|         20.6|       76.5|           2|
|       2| 2024-10-01 00:18:38|         7.42|       33.1|           4|
|       2| 2024-10-01 00:20:06|        19.96|       70.0|           1|
|       1| 2024-10-01 00:09:02|          2.6|       15.6|           1|
+--------+--------------------+-------------+-----------+------------+
only showing top 5 rows


## Questão 3

Nesta questão, serão filtradas as corridas com distância superior a 5 milhas e com 3 ou mais passageiros. Em seguida, será exibida a quantidade de corridas que atendem simultaneamente a essas condições.

In [21]:
# Filtrar as corridas conforme as condições solicitadas

corridas_filtradas = df.filter(
    (df.trip_distance > 5) &
    (df.passenger_count >= 3)
)

corridas_filtradas.count()

50665

## Questão 4

Quando utilizamos `inferSchema=True`, o Spark analisa os dados do arquivo para tentar identificar automaticamente o tipo de cada coluna, como inteiro, decimal, texto ou data.

Essa opção facilita o carregamento dos dados, pois não é necessário informar manualmente o tipo de cada coluna. Porém, o Spark precisa analisar os dados para realizar essa inferência, o que pode aumentar o tempo de leitura em arquivos muito grandes. Além disso, dependendo dos valores existentes no arquivo, o tipo de uma coluna pode ser interpretado de forma diferente do esperado.

Outra opção é definir o schema manualmente utilizando `StructType` e `StructField`. Nesse caso, informamos previamente o nome de cada coluna e o seu respectivo tipo de dado.

A definição manual do schema é mais controlada e pode ser mais eficiente para grandes volumes de dados, pois o Spark não precisa descobrir os tipos das colunas. Porém, exige que a estrutura do arquivo seja conhecida previamente. Caso algum tipo seja definido de forma incorreta, podem ocorrer erros ou valores nulos durante a leitura.

Portanto, o `inferSchema=True` é mais simples e prático, enquanto o schema manual oferece maior controle e previsibilidade, sendo especialmente útil quando trabalhamos com arquivos muito grandes e cuja estrutura já é conhecida.

## Questão 5

Nesta questão, as corridas serão agrupadas por tipo de pagamento (`payment_type`). Para cada grupo, será calculada a quantidade de corridas e a receita total, utilizando a soma da coluna `total_amount`. O resultado será exibido em ordem decrescente de receita.

In [22]:
from pyspark.sql import functions as F

resultado_q5 = (
    df.groupBy("payment_type")
    .agg(
        F.count("*").alias("quantidade_corridas"),
        F.sum("total_amount").alias("receita_total")
    )
    .orderBy(F.desc("receita_total"))
)

resultado_q5.show()

+------------+-------------------+--------------------+
|payment_type|quantidade_corridas|       receita_total|
+------------+-------------------+--------------------+
|           1|            3045849| 9.116799616010016E7|
|           2|             553536|1.2987084559999354E7|
|           0|             410746|1.0123049400000528E7|
|           3|              29100|  220775.24999999974|
|           4|              79511|  133192.01999999984|
|           5|                  1|                62.0|
+------------+-------------------+--------------------+



## Questão 6

Nesta questão, será criada a coluna `hora_embarque`, extraindo a hora da coluna `tpep_pickup_datetime`. Em seguida, as corridas serão agrupadas por hora para calcular a tarifa média (`fare_amount`) e a distância média (`trip_distance`) de cada período do dia.

In [23]:
# Criar a coluna com a hora do embarque e calcular as médias por hora

df_hora = df.withColumn(
    "hora_embarque",
    F.hour("tpep_pickup_datetime")
)

resultado_q6 = (
    df_hora.groupBy("hora_embarque")
    .agg(
        F.avg("fare_amount").alias("tarifa_media"),
        F.avg("trip_distance").alias("distancia_media")
    )
    .orderBy("hora_embarque")
)

resultado_q6.show(24)

+-------------+------------------+------------------+
|hora_embarque|      tarifa_media|   distancia_media|
+-------------+------------------+------------------+
|            0| 19.72867660335703| 5.130178643081671|
|            1| 17.54847894641617| 3.739999483030465|
|            2|16.426538461538446| 4.542445678033303|
|            3| 17.24064640950263| 3.401675265462839|
|            4| 22.33585451861572|11.412191651631977|
|            5|26.226564065583663| 23.33988120540463|
|            6|21.931859821807333|14.540589393296582|
|            7| 19.33026953083623|11.087329050022918|
|            8|18.511148615351107| 8.533842520592145|
|            9|18.400291333656767| 5.604490502277248|
|           10| 18.56454835768904|4.5114450807098905|
|           11|18.851552824117647| 4.075729557436569|
|           12|19.207714073999153| 4.468694683646846|
|           13| 19.95819012628161| 5.262006204074442|
|           14|20.604332332373676| 4.622973056355365|
|           15| 20.757107834

## Questão 7

No Spark, as **transformações (transformations)** são operações que criam um novo DataFrame a partir de outro, mas não executam o processamento imediatamente. Exemplos utilizados nas questões anteriores são `select()`, `filter()` e `groupBy()`.

Já as **ações (actions)** são operações que realmente fazem o Spark executar o processamento e retornar um resultado. Exemplos utilizados foram `show()` e `count()`.

O Spark utiliza o conceito de **lazy evaluation**, ou avaliação preguiçosa, porque ele não executa cada transformação no momento em que ela é declarada. Em vez disso, ele registra as operações que precisam ser feitas e só executa o processamento quando uma ação é chamada.

A vantagem prática desse comportamento é que o Spark consegue analisar toda a sequência de operações antes da execução e otimizar o plano de processamento. Isso pode reduzir operações desnecessárias e melhorar o desempenho, principalmente quando se trabalha com grandes volumes de dados.

## Questão 8

Nesta questão, serão consideradas apenas as corridas com `total_amount` maior que zero. Em seguida, será criada a coluna `percentual_gorjeta`, calculada pela razão entre `tip_amount` e `total_amount`, multiplicada por 100. Por fim, serão exibidas as 10 corridas com maior percentual de gorjeta.

In [24]:
# Calcular o percentual de gorjeta e exibir as 10 maiores

resultado_q8 = (
    df.filter(df.total_amount > 0)
    .withColumn(
        "percentual_gorjeta",
        (F.col("tip_amount") / F.col("total_amount")) * 100
    )
    .select(
        "VendorID",
        "total_amount",
        "tip_amount",
        "percentual_gorjeta"
    )
    .orderBy(F.desc("percentual_gorjeta"))
)

resultado_q8.show(10)

+--------+------------+----------+------------------+
|VendorID|total_amount|tip_amount|percentual_gorjeta|
+--------+------------+----------+------------------+
|       2|        1.63|      5.27| 323.3128834355828|
|       2|        2.07|      3.68| 177.7777777777778|
|       2|         1.6|      2.82|176.24999999999997|
|       2|        2.33|      3.72|159.65665236051504|
|       2|        2.54|      3.76|148.03149606299212|
|       2|        3.76|      3.96|105.31914893617022|
|       2|        39.7|      40.0|100.75566750629723|
|       2|        0.08|      0.08|             100.0|
|       1|       197.0|     196.0| 99.49238578680203|
|       1|       150.0|     149.0| 99.33333333333333|
+--------+------------+----------+------------------+
only showing top 10 rows


## Questão 9

Nesta questão, será utilizada a tabela de referência de zonas da NYC TLC para relacionar o código `PULocationID` ao respectivo bairro (`Borough`) de origem das corridas.

Após o carregamento da tabela de zonas, será realizado um `join` entre os DataFrames, seguido do agrupamento por bairro e da contagem da quantidade de corridas originadas em cada local.

In [25]:
# Baixar e carregar a tabela de zonas

!wget -q https://huggingface.co/datasets/alexvaroz/nyc_tripdata_2024_sample_4M/resolve/main/taxi_zone_lookup.csv

zonas = spark.read.csv(
    "taxi_zone_lookup.csv",
    header=True,
    inferSchema=True
)

zonas.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [26]:
# a) Fazer o join entre as corridas e a tabela de zonas

df_com_zonas = df.join(
    zonas,
    df.PULocationID == zonas.LocationID,
    "left"
)

In [27]:
# b) Agrupar por Borough e contar a quantidade de corridas

resultado_q9 = (
    df_com_zonas.groupBy("Borough")
    .agg(
        F.count("*").alias("quantidade_corridas")
    )
)

In [28]:
# c) Ordenar do bairro com mais corridas para o com menos

resultado_q9 = resultado_q9.orderBy(
    F.desc("quantidade_corridas")
)

resultado_q9.show()

+-------------+-------------------+
|      Borough|quantidade_corridas|
+-------------+-------------------+
|    Manhattan|            3641752|
|       Queens|             388736|
|     Brooklyn|              60200|
|        Bronx|              12702|
|      Unknown|              12172|
|          N/A|               2421|
|          EWR|                565|
|Staten Island|                195|
+-------------+-------------------+



## Questão 10

Nesta questão, será comparado o tempo de execução de uma operação simples de contagem (`count()`) com uma operação de agrupamento (`groupBy()`), analisando o impacto do processo de `shuffle` no desempenho do Spark.

In [29]:
import time

# Medir o tempo do count()
inicio = time.perf_counter()

total_registros = df.count()

fim = time.perf_counter()

tempo_count = fim - inicio

print(f"Total de registros: {total_registros}")
print(f"Tempo do count(): {tempo_count:.2f} segundos")

Total de registros: 4118743
Tempo do count(): 2.67 segundos


In [30]:
# Medir o tempo da operação de agrupamento da Questão 5

inicio = time.perf_counter()

resultado_groupby = (
    df.groupBy("payment_type")
    .agg(
        F.count("*").alias("quantidade_corridas"),
        F.sum("total_amount").alias("receita_total")
    )
    .collect()
)

fim = time.perf_counter()

tempo_groupby = fim - inicio

print(f"Tempo do groupBy(): {tempo_groupby:.2f} segundos")

Tempo do groupBy(): 9.75 segundos


### Análise dos resultados

Nesta execução, o `count()` levou aproximadamente **2,60 segundos**, enquanto a operação com `groupBy()` levou aproximadamente **10,31 segundos**.

O `groupBy()` foi mais demorado porque operações de agrupamento exigem que o Spark reorganize os dados de acordo com a chave utilizada no agrupamento. Esse processo é chamado de **shuffle**.

Durante o shuffle, registros que estão distribuídos em diferentes partições precisam ser redistribuídos para que os valores pertencentes ao mesmo grupo fiquem juntos e possam ser agregados. Essa movimentação aumenta o custo de processamento, pois envolve mais operações de leitura, escrita e transferência de dados entre partições.

Já operações mais simples, como filtragem ou seleção de colunas, normalmente podem ser executadas diretamente em cada partição, sem a necessidade de reorganizar todo o conjunto de dados.

Por isso, mesmo processando o mesmo volume de dados, operações como `groupBy()` tendem a ser mais custosas do que operações de contagem, filtragem ou seleção.